# Run the CLIP + LogReg baseline on the **real** VCR data

Use this to produce the actual validation numbers for your report.

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type → T4 GPU*.

You must first register at https://visualcommonsense.com/download/ and accept
the license. That page gives you download links for two files:
`vcr1annots.zip` (small) and `vcr1images.zip` (~30 GB). Paste those links in
Step 2 below.

## 1. Clone the repo and install

In [ ]:
!git clone https://github.com/Utkarzzzz/vcr-project.git
%cd vcr-project
!pip -q install -r requirements.txt

## 2. Download and unzip the dataset

Paste the links you got after registering. The images file is large; the
download + unzip can take 15–30 minutes on Colab.

In [ ]:
ANNOTS_URL = 'PASTE_vcr1annots.zip_LINK_HERE'
IMAGES_URL = 'PASTE_vcr1images.zip_LINK_HERE'

import os
os.makedirs('data', exist_ok=True)
!wget -q -O data/annots.zip "$ANNOTS_URL"
!wget -q -O data/images.zip "$IMAGES_URL"
!cd data && unzip -q annots.zip && unzip -q images.zip
!ls data

**Alternative — mount Google Drive.** If the copied links 403 inside Colab (the token can be tied to your browser session), download both zips through your browser, upload them once to a `VCR/` folder in your Google Drive, then run this cell instead of the download cell above.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('data', exist_ok=True)
# adjust the path if you named the Drive folder differently
!cp '/content/drive/MyDrive/VCR/vcr1annots.zip' data/annots.zip
!cp '/content/drive/MyDrive/VCR/vcr1images.zip' data/images.zip
!cd data && unzip -q annots.zip && unzip -q images.zip
!ls data

After unzipping you should have `data/train.jsonl`, `data/val.jsonl`,
`data/test.jsonl` and a `data/vcr1images/` folder. If the jsonl files landed in
a subfolder, move them into `data/` so the paths match.

## 3. Build features on a subset

The full train split is ~213k samples. Start with a subset so this finishes in
minutes; raise `--limit` later if you want higher accuracy.

In [ ]:
!python src/build_features.py --jsonl data/train.jsonl --out data/train --limit 4000
!python src/build_features.py --jsonl data/val.jsonl   --out data/val   --limit 1000

## 4. Train and evaluate — these are the numbers for your report

In [ ]:
!python src/train.py    --train data/train
!python src/evaluate.py --features data/val

## 5. Generate test-set predictions (deliverable)

In [ ]:
!python src/predict.py --jsonl data/test.jsonl --out predictions.json --limit 5000
from google.colab import files
files.download('predictions.json')